# Tool Calling

In [2]:
import dotenv
from agents import Agent, ModelSettings, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [3]:
def get_food_calories(food_item: str) -> str:
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Simple calorie database - in real world, you'd use USDA API
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }

    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    else:
        return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, chicken breast, or rice."

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `get_food_calories` function_

In [4]:
get_food_calories('banana')

'Banana: 105 calories per medium banana (118g)'

In [11]:
calorie_agent = Agent (
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories],
    )

teaching_agent =Agent(
    name ="teaching assistent",
    instructions="""
    your subject matter expert and will give out advice on 
    subject EVS - Environment studies 
    """,
    tools=[lookup_evs_concept] ,
   )

In [12]:
with trace("EVS Assistant with tools"):
    result = await Runner.run(
        teaching_agent, "what is photosynthesis?"
    )
    print(result.final_output)

AttributeError: 'function' object has no attribute 'name'

Enforce tools use:

In [ ]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories],
    model_settings=ModelSettings(tool_choice="get_food_calories"),
)

with trace("Nutrition Assistant with tools enforced"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

AttributeError: 'function' object has no attribute 'name'

In [6]:
def lookup_evs_concept(concept: str) -> str:
    """Look up a concise explanation of a core EVS concept."""
    concepts = {
        "photosynthesis": (
            "Photosynthesis is the process by which green plants, algae, and some "
            "bacteria use sunlight to convert carbon dioxide and water into sugars, "
            "releasing oxygen as a by-product."
        ),
        "ozone depletion": (
            "Ozone depletion is the thinning of the stratospheric ozone layer, "
            "mainly caused by ozone-depleting substances such as CFCs. It allows "
            "more harmful ultraviolet radiation to reach Earth's surface."
        ),
        "global warming": (
            "Global warming is the long-term rise in Earth's average surface "
            "temperature, primarily driven by human-caused greenhouse gas emissions."
        ),
        "greenhouse effect": (
            "The greenhouse effect is the warming that occurs when gases in the "
            "atmosphere absorb and re-emit heat. It is a natural process, but "
            "increased greenhouse gases strengthen it and contribute to global warming."
        ),
        
    }
    aliases = {
        "ozon deplition": "ozone depletion",
        "ozone deplition": "ozone depletion",
        "green house effect": "greenhouse effect",
    }

    key = " ".join(concept.lower().split())
    key = aliases.get(key, key)
    if key in concepts:
        return f"{key.title()}: {concepts[key]}"

    available = ", ".join(concepts)
    return f"I don't have information about '{concept}'. Try one of these concepts: {available}."


evs_agent = Agent(
    name="EVS Assistant",
    instructions="""
    You are a helpful Environmental Studies (EVS) assistant.
    Use the lookup tool to explain the requested concept accurately and concisely.
    If asked about multiple listed concepts, look up each one.
    """,
    tools=[lookup_evs_concept],
    model_settings=ModelSettings(tool_choice="lookup_evs_concept"),
)

In [7]:
for concept in (
    "photosynthesis",
    "ozon deplition",
    "global warming",
    "green house effect",
):
    print(lookup_evs_concept(concept))

Photosynthesis: Photosynthesis is the process by which green plants, algae, and some bacteria use sunlight to convert carbon dioxide and water into sugars, releasing oxygen as a by-product.
Ozone Depletion: Ozone depletion is the thinning of the stratospheric ozone layer, mainly caused by ozone-depleting substances such as CFCs. It allows more harmful ultraviolet radiation to reach Earth's surface.
Global Warming: Global warming is the long-term rise in Earth's average surface temperature, primarily driven by human-caused greenhouse gas emissions.
Greenhouse Effect: The greenhouse effect is the warming that occurs when gases in the atmosphere absorb and re-emit heat. It is a natural process, but increased greenhouse gases strengthen it and contribute to global warming.


## Math Tutor Agent

In [8]:
math_tutor = Agent(
    name="Math Tutor",
    instructions="""
    You are a patient, encouraging math tutor for learners at different levels.
    First identify what the student is trying to understand and adapt your
    explanation to their level. Help them reason through problems with clear,
    manageable steps, useful hints, and guiding questions instead of jumping
    straight to the answer. If the student asks for a complete solution, show
    the steps and explain why they work. Use correct mathematical notation,
    check calculations, and invite the student to try the next step.
    """,
)


In [9]:
with trace("Math Tutor"):
    result = await Runner.run(
        math_tutor,
        "Can you help me solve 2x + 5 = 13? Give me a hint first.",
    )
    print(result.final_output)

Hint: What operation can you use to remove the 5 on the left side? Try applying the same operation to both sides of the equation.


In [10]:
import math


def cylinder_surface_area(radius: float, height: float) -> float:
    """Return the cylinder's total surface area in square units."""
    if radius < 0 or height < 0:
        raise ValueError("Radius and height must be non-negative.")

    return 2 * math.pi * radius * (radius + height)


cylinder_surface_area(3, 5)

150.79644737231007